In [44]:
!pip install optuna
!pip install kagglehub

In [45]:
!pip install optuna mlflow dagshub

In [46]:
import mlflow
import os
from getpass import getpass
import optuna
import kagglehub
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Input
from tensorflow.keras.optimizers import RMSprop, SGD, Adam
from tensorflow.keras import regularizers
from keras.callbacks import ModelCheckpoint, EarlyStopping
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_slice
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank


In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [48]:
REPO_NAME= "TAREA2_credit_card_fraud"
REPO_OWNER= "EstefaniaLima"  #
USER_NAME = "EstefaniaLima" #Escribir su usuario

In [49]:
os.environ['MLFLOW_TRACKING_USERNAME'] = USER_NAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = getpass('Ingresa tu token de DAGsHub: ')


Ingresa tu token de DAGsHub: ··········


In [50]:
# Le decimos a MLflow a donde enviar los datos
mlflow.set_tracking_uri(f'https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow')
# Activamos el guardado automático de Keras y creamos el experimento
mlflow.tensorflow.autolog()
mlflow.set_experiment("deteccion_fraude_redes_neuronales")

2026/06/28 00:01:15 INFO mlflow.tracking.fluent: Experiment with name 'deteccion_fraude_redes_neuronales' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/019fa0560d2647ef82a3e98b53bc685a', creation_time=1782604876060, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1782604876060, lifecycle_stage='active', name='deteccion_fraude_redes_neuronales', tags={}, trace_location=None, workspace='default'>

In [51]:
#Cargamos el dataset

def cargar_y_preprocesar_datos():
    print("Descargando dataset")
    # Descarga KaggleHub
    path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

    print(f"Dataset descargado en: {path}")
    df = pd.read_csv(f"{path}/creditcard.csv")

    X = df.drop(columns=['Class'])
    y = df['Class'].values

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train[['Time', 'Amount']] = scaler.fit_transform(X_train[['Time', 'Amount']])
    X_val[['Time', 'Amount']] = scaler.transform(X_val[['Time', 'Amount']])

    return X_train.values, X_val.values, y_train, y_val



In [52]:
#pesos de clase
def calcular_pesos_clase(y_train):
    negativos, positivos = np.bincount(y_train)
    total = negativos + positivos
    peso_0 = (1 / negativos) * (total / 2.0)
    peso_1 = (1 / positivos) * (total / 2.0)
    return {0: peso_0, 1: peso_1}

In [53]:
# Definimos la función create_model
def create_model(trial,dim):

    model = models.Sequential()
    #Opciones número de capas
    n_layers = trial.suggest_int("n_layers", 1, 3)

    #Optuna nos ayuda a elegir la función de activación
    activacion_opciones = ["relu", "tanh", "swish"]
    activacion_seleccionada=trial.suggest_categorical("activacion",activacion_opciones)

    # Optuna sugiere: Número de unidades por capa

    for i in range(n_layers):

        n_units = trial.suggest_categorical(f"n_units_l{i}", [32, 64, 128, 256])


        dropout_rate = trial.suggest_float(f"dropout_l{i}", 0.1, 0.5)


        if i == 0:
            model.add(layers.Dense(n_units, activation=activacion_seleccionada, input_shape=(dim,)))
        else:
            model.add(layers.Dense(n_units, activation=activacion_seleccionada))

        model.add(layers.Dropout(dropout_rate))

    # CAPA DE SALIDA
    model.add(layers.Dense(1, activation='sigmoid'))

    # Optuna sugiere: Valor de learning rate

    lr = trial.suggest_float("lr", 1e-4, 1e-1, log = True)

    # Optuna sugiere: Optimizador

    optimizer_options = ["sgd", "adam", "rmsprop"]

    optimizer_selected = trial.suggest_categorical("optimizer", optimizer_options)

    if optimizer_selected == "adam":

        optimizer = tf.keras.optimizers.Adam(learning_rate = lr)

    elif optimizer_selected == "sgd":

        optimizer = tf.keras.optimizers.SGD(learning_rate = lr)

    elif optimizer_selected == "rmsprop":

        optimizer = tf.keras.optimizers.RMSprop(learning_rate = lr)

    #Aquí se compilamos el modelo

    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=[keras.metrics.AUC(name='auc_pr', curve='PR'), 'accuracy'])

    return model




In [54]:
#Definimos una función objrtivo con registro en MLFLOW
def objective(trial, X_train, y_train, X_val, y_val, pesos):
    with mlflow.start_run(nested=True):
        modelo = create_model(trial, dim=X_train.shape[1])
        mlflow.log_params(trial.params)

        earlystop = keras.callbacks.EarlyStopping(
            monitor='val_auc_pr', mode='max', patience=5, restore_best_weights=True
        )

        historial = modelo.fit(
            X_train, y_train,
            epochs=20,
            batch_size=trial.suggest_categorical("batch_size", [1024, 2048]),
            validation_data=(X_val, y_val),
            class_weight=pesos,
            callbacks=[earlystop],
            verbose=0
        )

        best_val_auc_pr = max(historial.history['val_auc_pr'])
        mlflow.log_metric("best_val_auc_pr", best_val_auc_pr)

        best_val_acc = max(historial.history['val_accuracy'])
        mlflow.log_metric("best_val_accuracy", best_val_acc)

        best_val_loss = min(historial.history['val_loss'])
        mlflow.log_metric("best_val_loss", best_val_loss)

        return max(historial.history['val_auc_pr'])

In [55]:
if __name__ == "__main__":
    X_train, X_val, y_train, y_val = cargar_y_preprocesar_datos()
    pesos = calcular_pesos_clase(y_train)

    study = optuna.create_study(direction="maximize")
    study.optimize(lambda trial: objective(trial, X_train, y_train, X_val, y_val, pesos), n_trials=40)

    print(f"Mejor AUC-PR: {study.best_value:.4f}")

    fig1 = plot_optimization_history(study)
    fig1.show()
    fig2 = plot_param_importances(study)
    fig2.show()

Descargando dataset
Using Colab cache for faster access to the 'creditcardfraud' dataset.
Dataset descargado en: /kaggle/input/creditcardfraud


[I 2026-06-28 00:01:22,393] A new study created in memory with name: no-name-a44fff1f-7386-4d22-b9c2-91e00a4572ed
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step


2026/06/28 00:02:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run redolent-snake-762 at: https://dagshub.com/EstefaniaLima/TAREA2_credit_card_fraud.mlflow/#/experiments/0/runs/9a1dab92ae66401fa2bb053003c244e8
🧪 View experiment at: https://dagshub.com/EstefaniaLima/TAREA2_credit_card_fraud.mlflow/#/experiments/0


...

🏃 View run big-bee-66 at: https://dagshub.com/EstefaniaLima/TAREA2_credit_card_fraud.mlflow/#/experiments/0/runs/66b49c24caed415d8481697f60f62a6c
🧪 View experiment at: https://dagshub.com/EstefaniaLima/TAREA2_credit_card_fraud.mlflow/#/experiments/0


[I 2026-06-28 01:34:52,791] Trial 38 finished with value: 0.730772852897644 and parameters: {'n_layers': 3, 'activacion': 'relu', 'n_units_l0': 64, 'dropout_l0': 0.17147885230960996, 'n_units_l1': 256, 'dropout_l1': 0.1019069115544429, 'n_units_l2': 64, 'dropout_l2': 0.16585633937495323, 'lr': 0.0009417049047759136, 'optimizer': 'adam', 'batch_size': 1024}. Best is trial 0 with value: 0.7511404752731323.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step


2026/06/28 01:36:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[I 2026-06-28 01:36:53,071] Trial 39 finished with value: 0.7016025185585022 and parameters: {'n_layers': 3, 'activacion': 'swish', 'n_units_l0': 256, 'dropout_l0': 0.2058141248144506, 'n_units_l1': 32, 'dropout_l1': 0.28828194213152103, 'n_units_l2': 32, 'dropout_l2': 0.3224091219978858, 'lr': 0.019472061737088646, 'optimizer': 'sgd', 'batch_size': 2048}. Best is trial 0 with value: 0.7511404752731323.


🏃 View run invincible-swan-626 at: https://dagshub.com/EstefaniaLima/TAREA2_credit_card_fraud.mlflow/#/experiments/0/runs/1aa9b7c7f672485abdd98c93f18f5b26
🧪 View experiment at: https://dagshub.com/EstefaniaLima/TAREA2_credit_card_fraud.mlflow/#/experiments/0
Mejor AUC-PR: 0.7511


In [67]:
trial = study.best_trial

print("Mejor intento: ", trial)

print("Valor: ", trial.value)
print("Hiperparámetros: ", trial.params)

Mejor intento:  FrozenTrial(number=0, state=<TrialState.COMPLETE: 1>, values=[0.7511404752731323], datetime_start=datetime.datetime(2026, 6, 28, 0, 1, 22, 396168), datetime_complete=datetime.datetime(2026, 6, 28, 0, 2, 46, 556754), params={'n_layers': 2, 'activacion': 'tanh', 'n_units_l0': 256, 'dropout_l0': 0.11014093412032461, 'n_units_l1': 32, 'dropout_l1': 0.3200102614536141, 'lr': 0.029750467269739508, 'optimizer': 'sgd', 'batch_size': 2048}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_layers': IntDistribution(high=3, log=False, low=1, step=1), 'activacion': CategoricalDistribution(choices=('relu', 'tanh', 'swish')), 'n_units_l0': CategoricalDistribution(choices=(32, 64, 128, 256)), 'dropout_l0': FloatDistribution(high=0.5, log=False, low=0.1, step=None), 'n_units_l1': CategoricalDistribution(choices=(32, 64, 128, 256)), 'dropout_l1': FloatDistribution(high=0.5, log=False, low=0.1, step=None), 'lr': FloatDistribution(high=0.1, log=True, low=0.0001, st

In [68]:
plot_parallel_coordinate(study)

In [69]:
plot_slice(study)

In [63]:
plot_rank(study)